# Lab 02: From a Data File to a Chart with Python and JavaScript

**Name:** Usman Hassan  
**Roll number:** 1421-324032    
**Date:** 26 October 2026  
**AI tools used:** Claude (Anthropic)

## Task 1
Set up: load the data with pandas, print its shape, and check Node.js.


In [24]:
import pandas as pd

df = pd.read_csv("students_marks.csv")
print(df.shape)

(25, 8)


In [25]:
!node --version

v20.19.0


## Task 2
Summarise with pandas: section averages, top 3 students, and the best section in Python.

In [26]:
subjects = ["Math", "Stats", "Python", "Databases", "English"]

# average of every subject for each section
summary = df.groupby("Section")[subjects].mean().round(2).reset_index()
print(summary)

  Section   Math  Stats  Python  Databases  English
0       A  71.10  71.30   76.00       73.4    69.10
1       B  68.25  69.17   71.33       68.5    64.25
2       C  74.33  69.67   73.00       76.0    65.00


In [27]:
# average of every student (axis=1 = across the row)
df["Average"] = df[subjects].mean(axis=1).round(2)

top3 = df.sort_values("Average", ascending=False)[["Name", "Section", "Average"]].head(3)
print(top3)

           Name Section  Average
19  Rabia Aslam       B    100.0
17   Laiba Khan       C     89.6
7   Fatima Noor       A     85.4


In [28]:
# section with the highest average in Python
best_py = summary.loc[summary["Python"].idxmax()]
print("Highest Python average:", "Section", best_py["Section"], "with", best_py["Python"])

Highest Python average: Section A with 76.0


## Task 3
Write the results as JSON (orient="records") and print summary.json.

In [29]:
summary.to_json("summary.json", orient="records")
df.to_json("students.json", orient="records")   # df now includes the Average column

print(open("summary.json").read())

[{"Section":"A","Math":71.1,"Stats":71.3,"Python":76.0,"Databases":73.4,"English":69.1},{"Section":"B","Math":68.25,"Stats":69.17,"Python":71.33,"Databases":68.5,"English":64.25},{"Section":"C","Math":74.33,"Stats":69.67,"Python":73.0,"Databases":76.0,"English":65.0}]


## Task 4
JavaScript (`t4.js`) reads both JSON files and answers four questions using filter, map, sort and forEach.

In [30]:
%%writefile t4.js
const fs = require("fs");
const students = JSON.parse(fs.readFileSync("students.json", "utf8"));
const summary = JSON.parse(fs.readFileSync("summary.json", "utf8"));

// 1. number of students
console.log("Number of students:", students.length);

// 2. students whose Average is below 60 (filter + map)
const below60 = students.filter(s => s.Average < 60).map(s => s.Name);
console.log("Average below 60:", below60);

// 3. section with the highest Python average (sort a copy)
const bestPython = [...summary].sort((a, b) => b.Python - a.Python)[0];
console.log("Best in Python:", "Section", bestPython.Section, bestPython.Python);

// 4. one line per section (forEach + template literal)
summary.forEach(s => {
  console.log(`Section ${s.Section}: Math ${s.Math}, Python ${s.Python}`);
});

Overwriting t4.js


In [31]:
!node t4.js

Number of students: 25
Average below 60: [
  'Hina Malik',
  'Hamza Raza',
  'Asad Mirza',
  'Faisal Qureshi',
  'Mahnoor Zafar'
]
Best in Python: Section A 76
Section A: Math 71.1, Python 76
Section B: Math 68.25, Python 71.33
Section C: Math 74.33, Python 73


## Task 5
Draw **Average Python by section** with JavaScript, show it in Colab and save `chart.html`.

In [32]:
%%writefile chart.js
// data comes from Python (summary.json)
const base = 250;                       // y of the x-axis
let svg = `<svg width="460" height="300" font-family="Arial">`;
svg += `<text x="230" y="24" text-anchor="middle"` +
       ` font-weight="bold">Average Python by section</text>`;
svg += `<line x1="40" y1="${base}" x2="440" y2="${base}"` +
       ` stroke="black"/>`;
data.forEach((d, i) => {
  const h = d.Python * 2;               // 100 marks = 200 px
  const x = 60 + i * 120;               // left edge of the bar
  svg += `<rect x="${x}" y="${base - h}" width="80"` +
         ` height="${h}" fill="#C2410C"/>`;
  svg += `<text x="${x + 40}" y="${base - h - 6}"` +
         ` text-anchor="middle">${d.Python}</text>`;
  svg += `<text x="${x + 40}" y="${base + 20}"` +
         ` text-anchor="middle">Section ${d.Section}</text>`;
});
svg += `</svg>`;
document.getElementById("chart").innerHTML = svg;

Overwriting chart.js


In [33]:
from IPython.display import HTML, display

data_json = summary.to_json(orient="records")
js = open("chart.js").read()
page = (f'<div id="chart"></div>'
        f'<script>const data = {data_json};\n{js}</script>')
display(HTML(page))
open("chart.html", "w").write(page)

1229

In [34]:
#  download chart.html straight to your computer, then double-click it to open in Chrome
from google.colab import files
files.download("chart.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Task 6
**Comparison of the two charts**

The Power BI chart was faster to make because I only dragged Section and Python into the chart and changed Sum to Average, while the JavaScript chart needed code for every bar, label and the axis. In JavaScript I can control every pixel (exact bar positions, sizes, colours and text), but Power BI gives me interactivity like the Section slicer and the Students card without writing any code.